In [4]:
# 02_FEATURE_ENG_AND_MERGE: Building the Master Dataset

import pandas as pd
import os

print("Loading cleaned datasets (This may take a minute)...")
df_enrol = pd.read_csv('../data/processed/cleaned_aadhar_enrolment_merged.csv', parse_dates=['date'])
df_bio = pd.read_csv('../data/processed/cleaned_aadhar_biometric_merged.csv', parse_dates=['date'])
df_demo = pd.read_csv('../data/processed/cleaned_aadhar_demographic_merged.csv', parse_dates=['date'])

# Define our universal join keys
JOIN_KEYS = ['date', 'state', 'district', 'pincode']

print("✅ Data loaded. Ready for aggregation and merge.")

Loading cleaned datasets (This may take a minute)...
✅ Data loaded. Ready for aggregation and merge.


In [5]:
print("1. Aggregating datasets to fix duplicate daily center batches...")

# Group by our keys and sum the numbers to squash duplicates
df_enrol_agg = df_enrol.groupby(JOIN_KEYS, as_index=False).sum()
df_bio_agg = df_bio.groupby(JOIN_KEYS, as_index=False).sum()
df_demo_agg = df_demo.groupby(JOIN_KEYS, as_index=False).sum()

print("2. Executing Master Merge (Outer Join)...")

# Merge Enrolment and Demographic
merged_temp = pd.merge(df_enrol_agg, df_demo_agg, on=JOIN_KEYS, how='outer')

# Merge the result with Biometric
df_master = pd.merge(merged_temp, df_bio_agg, on=JOIN_KEYS, how='outer')

# Handle the NaNs generated by the Outer Join
numeric_cols = df_master.select_dtypes(include=['float64']).columns
df_master[numeric_cols] = df_master[numeric_cols].fillna(0).astype(int)

print(f"Master Dataset Shape: {df_master.shape}")
print("✅ Merge successful!")

1. Aggregating datasets to fix duplicate daily center batches...
2. Executing Master Merge (Outer Join)...
Master Dataset Shape: (2310875, 11)
✅ Merge successful!


In [6]:
print("--- DATA INTEGRITY CHECK ---")

# We must compare against the RAW dataframe
raw_infant_sum = df_enrol['age_0_5'].fillna(0).sum()
master_infant_sum = df_master['age_0_5'].sum()

print(f"Original Infant Enrolments: {raw_infant_sum:,.0f}")
print(f"Master Infant Enrolments:   {master_infant_sum:,.0f}")

if raw_infant_sum == master_infant_sum:
    print("✅ INTEGRITY PASSED: Data perfectly preserved. No duplication!")
else:
    print("❌ ERROR: Sums still do not match.")

--- DATA INTEGRITY CHECK ---
Original Infant Enrolments: 3,546,965
Master Infant Enrolments:   3,546,965
✅ INTEGRITY PASSED: Data perfectly preserved. No duplication!


In [7]:
print("Engineering new features...")

# 1. Temporal Features (Makes Tableau filtering much faster)
df_master['month'] = df_master['date'].dt.month
df_master['quarter'] = df_master['date'].dt.quarter

# 2. Aggregation Features (Total workload per row)
# Ensure columns exist before summing (in case of spelling differences)
df_master['total_enrolments'] = df_master.get('age_0_5', 0) + df_master.get('age_5_17', 0) + df_master.get('age_18_greater', 0)
df_master['total_demo_updates'] = df_master.get('demo_age_5_17', 0) + df_master.get('demo_age_17_', 0)
df_master['total_bio_updates'] = df_master.get('bio_age_5_17', 0) + df_master.get('bio_age_17_', 0)

# 3. The "Hardware Bottleneck" Ratio
# We add +1 to the denominator to prevent ZeroDivisionError
df_master['bio_to_demo_ratio'] = df_master['total_bio_updates'] / (df_master['total_demo_updates'] + 1)

print("✅ Features engineered successfully.")

Engineering new features...
✅ Features engineered successfully.


In [8]:
out_path = '../data/processed/master_aadhaar_2025.csv'

print("Exporting master dataset...")
df_master.to_csv(out_path, index=False)

print(f"🎉 SUCCESS: Master dataset exported to {out_path}")
print(f"Final File Size: {os.path.getsize(out_path) / (1024*1024):.2f} MB")

Exporting master dataset...
🎉 SUCCESS: Master dataset exported to ../data/processed/master_aadhaar_2025.csv
Final File Size: 166.03 MB


insights in reports/findings.md